# 04 · Functions — exercise solutions

Complete answers to all six exercises in [the student chapter](../notebooks/04_functions.ipynb). Run this notebook independently in a fresh Python 3 kernel. Every solution defines its own functions and test data; no other notebook needs to have run.

Compare each function's signature, documented contract, and return value with your version. Assertions cover ordinary results, boundary cases, and behavior such as preserving input objects or intentionally mutating a supplied list.


## Solution 04.1 · Clean a reading and return two values

String methods return a cleaned name without changing the original string. `float` accepts surrounding whitespace and signs. Returning a comma-separated pair creates a tuple that the caller can unpack. Letting invalid numeric text raise `ValueError` preserves the conversion's useful error signal.


In [ ]:
def parse_reading(name, raw_value):
    """Return (stripped title-case name, float value); invalid values raise ValueError."""
    return name.strip().title(), float(raw_value)

raw_name = "  air TEMPERATURE "
name, value = parse_reading(raw_name, " 21.5 ")
assert (name, value) == ("Air Temperature", 21.5)
assert isinstance(value, float)
assert raw_name == "  air TEMPERATURE "
assert parse_reading(" outside ", "-4.25") == ("Outside", -4.25)
assert parse_reading("zero", "0") == ("Zero", 0.0)
try:
    parse_reading("temperature", "warm")
except ValueError:
    print("Expected ValueError: warm is not numeric text.")
else:
    raise AssertionError("Invalid numeric text should fail")
print(name, value)
print("04.1 checks passed")


## Solution 04.2 · Add labels without accidental sharing

`None` identifies an omitted list, so a fresh list is created on each such call. Testing `is None` preserves a supplied empty list. Mutation is intentional when the caller supplies a list, and the docstring states that contract.


In [ ]:
def add_label(label, labels=None):
    """Append a stripped label and return labels, creating a list when omitted.

    An explicitly supplied list is mutated and returned unchanged in identity.
    Empty stripped labels are retained.
    """
    if labels is None:
        labels = []
    labels.append(label.strip())
    return labels

first = add_label(" A ")
second = add_label("B")
assert first == ["A"] and second == ["B"]
assert first is not second
existing = []
returned = add_label(" C ", existing)
assert returned is existing
assert existing == ["C"]
add_label("D", existing)
assert existing == ["C", "D"]
assert add_label("  ") == [""]
assert add_label(" E ", None) == ["E"]
assert first == ["A"] and second == ["B"]
print(first, second, existing)
print("04.2 checks passed")


## Solution 04.3 · A flexible total

`amounts` is a tuple of positional arguments. The parameters after `*amounts` can only be set by keyword. The empty sum is zero, so an empty call returns the offset. A final positional `4` contributes to the sum; it does not set the multiplier or offset.


In [ ]:
def adjusted_total(*amounts, multiplier=1, offset=0):
    """Return sum(amounts) * multiplier + offset; an empty sum is zero."""
    return sum(amounts) * multiplier + offset

assert adjusted_total(2, 3) == 5
assert adjusted_total(2, 3, multiplier=2, offset=1) == 11
assert adjusted_total() == 0
assert adjusted_total(offset=7) == 7
amounts = [1, 2, 3]
assert adjusted_total(*amounts) == 6
assert amounts == [1, 2, 3]
assert adjusted_total(-2, 3) == 1
assert adjusted_total(2, 3, multiplier=0.5) == 2.5
assert adjusted_total(2, 3, multiplier=0, offset=4) == 4
assert adjusted_total(2, 3, 4) == 9
options = {"multiplier": 2, "offset": 1}
assert adjusted_total(*[2, 3], **options) == 11
print(adjusted_total(2, 3, multiplier=2, offset=1))
print("04.3 checks passed")


## Solution 04.4 · Format an event with metadata

`**details` collects metadata. Sorting its items orders the lines by their unique string keys. Joining the lines avoids an unwanted trailing newline, including when no metadata is supplied. Unpacking a dictionary into keyword arguments does not mutate that dictionary.


In [ ]:
def format_event(event, **details):
    """Return event followed by alphabetically ordered 'key: value' lines."""
    lines = [event]
    for key, value in sorted(details.items()):
        lines.append(f"{key}: {value}")
    return "\n".join(lines)

expected = "Saved\ncount: 3\nuser: Ada"
assert format_event("Saved", user="Ada", count=3) == expected
assert format_event("Saved") == "Saved"
first_details = {"user": "Ada", "count": 3}
second_details = {"count": 3, "user": "Ada"}
assert format_event("Saved", **first_details) == format_event("Saved", **second_details)
assert first_details == {"user": "Ada", "count": 3}
assert list(first_details) == ["user", "count"]
assert format_event("Checked", valid=False, value=None) == "Checked\nvalid: False\nvalue: None"
assert format_event.__doc__ is not None
print(format_event("Saved", **first_details))
print("04.4 checks passed")


## Solution 04.5 · Supply the transformation

The transformation is a function argument, so the loop is reusable. `make_offset` returns an inner function that reads the offset captured by that particular factory call. Two returned functions can therefore retain different offsets.

The list comprehension creates a new outer list. A callback can still mutate a mutable element or other state; the no-mutation checks here use pure callbacks on immutable numbers.


In [ ]:
def transform_values(values, transform):
    """Return a new list of transformed values without modifying the input list.

    A supplied callback can have its own side effects; use a pure callback when
    the objects inside the input must also remain unchanged.
    """
    return [transform(value) for value in values]

def make_offset(offset):
    """Return a function that adds the captured offset to its argument."""
    def add_offset(value):
        return value + offset
    return add_offset

values = [1, 2, 3]
add_ten = make_offset(10)
subtract_two = make_offset(-2)
transformed = transform_values(values, add_ten)
assert transformed == [11, 12, 13]
assert values == [1, 2, 3]
assert transformed is not values
assert transform_values(values, str) == ["1", "2", "3"]
assert transform_values([], add_ten) == []
assert add_ten(5) == 15 and subtract_two(5) == 3
assert add_ten(0) == 10 and subtract_two(0) == -2
assert transform_values((1, 2), make_offset(0.5)) == [1.5, 2.5]
print(transformed, transform_values(values, str))
print("04.5 checks passed")


## Solution 04.6 · Summarize scores with a clear contract

Compute the count first so empty input can avoid division by zero. `None` explicitly means there is no mean, while zero counts are meaningful. Use `>=` so a score exactly at the threshold passes. The bare `*` makes `pass_mark` keyword-only.


In [ ]:
def summarize_scores(scores, *, pass_mark=50):
    """Return count, mean, and passing count for a sequence of numeric scores.

    Scores equal to pass_mark pass. Empty input has mean None and zero counts.
    The supplied sequence is not modified.
    """
    count = len(scores)
    mean = sum(scores) / count if count else None
    passed = sum(1 for score in scores if score >= pass_mark)
    return {"count": count, "mean": mean, "passed": passed}

scores = [40, 50, 90]
assert summarize_scores(scores) == {"count": 3, "mean": 60.0, "passed": 2}
assert summarize_scores(scores, pass_mark=80) == {"count": 3, "mean": 60.0, "passed": 1}
assert summarize_scores([]) == {"count": 0, "mean": None, "passed": 0}
assert summarize_scores([50]) == {"count": 1, "mean": 50.0, "passed": 1}
assert summarize_scores([49.9])["passed"] == 0
assert summarize_scores((0, 100), pass_mark=0) == {"count": 2, "mean": 50.0, "passed": 2}
assert scores == [40, 50, 90]
try:
    summarize_scores(scores, 80)
except TypeError:
    print("Expected TypeError: pass_mark is keyword-only.")
else:
    raise AssertionError("A positional pass_mark should fail")
print(summarize_scores(scores))
print("04.6 checks passed")


All six answers separate returned values from display output. Return to [the student chapter](../notebooks/04_functions.ipynb), or find the reusable functions in [examples/04_functions.py](../examples/04_functions.py).
